# LunarLander LTLf training with discretized SAC (DSAC)

This self-contained Kaggle notebook installs the dependencies, reconstructs the `sac/` framework, trains Stable-Baselines3 SAC and packages every output. SAC sees a one-dimensional continuous `Box(-1, 1)` action space; the action wrapper scales and floors each value into one of LunarLander's four original discrete actions. No Dataset or additional upload is required.

Enable a GPU from **Settings → Accelerator → GPU** before starting.


## 1. Install system and Python dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq mona graphviz swig
%pip install -q "gymnasium[box2d]" stable-baselines3 ltlf2dfa graphviz pandas matplotlib pillow


## 2. Create the writable project directory


In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/dsac")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")


## 3. Write the abstract MDP and LTLf automaton module


In [ ]:
%%writefile abstract_mdps.py
import re
from collections import defaultdict
import numpy as np

# Import the LTLf parser provided by ltlf2dfa.
from ltlf2dfa.parser.ltlf import LTLfParser
from graphviz import Source

class LTLfAutomaton:
    """
    Wrap ltlf2dfa and expose its DFA as a graph that can be traversed by the MDP.
    """
    def __init__(self, formula_str):
        self.formula_str = formula_str
        
        # Parse the formula and generate its DFA in DOT format.
        parser = LTLfParser()
        parsed_formula = parser(formula_str)
        dot_string = parsed_formula.to_dfa()
        self.dot_string = parsed_formula.to_dfa()
        
        # Initialize the automaton data structures.
        self.states = set()
        self.accepting_states = set()
        self.transitions = {}  # {source_state: [(Boolean_guard, destination_state), ...]}
        self.initial_state = None
        
        # Extract states and transitions from the DOT representation.
        self._parse_dot(dot_string)
        
        # Keep a stable state order for the MDP and one-hot encodings.
        self.states = sorted(list(self.states))
        self.num_phases = len(self.states)

    def _parse_dot(self, dot_string):
        """
        Parse the DOT output and extract states, accepting states, the initial
        state, and guarded transitions.
        """
        # Extract accepting states, e.g. node [shape = doublecircle]; 2 3;.
        match_acc = re.search(r'node\s*\[shape\s*=\s*doublecircle\]\s*;\s*(.*?);', dot_string)
        if match_acc:
            acc_str = match_acc.group(1).replace(',', ' ')
            self.accepting_states = set(int(s) for s in acc_str.split() if s.strip().isdigit())
            
        # Extract guarded transitions, e.g. 1 -> 2 [label="wp1 & ~wp2"].
        trans_matches = re.findall(r'(\d+)\s*->\s*(\d+)\s*\[label\s*=\s*"(.*?)"\]', dot_string)
        for src_str, dst_str, guard in trans_matches:
            src = int(src_str)
            dst = int(dst_str)
            self.states.add(src)
            self.states.add(dst)
            
            if src not in self.transitions:
                self.transitions[src] = []
            self.transitions[src].append((guard, dst))
            
        # Extract the initial state from the unlabeled edge leaving the invisible node.
        # Example: 0 [style=invis]; 0 -> 1;.
        init_match = re.search(r'(\d+)\s*->\s*(\d+)\s*;', dot_string)
        if init_match:
            self.initial_state = int(init_match.group(2))
        else:
            self.initial_state = min(self.states) if self.states else 0

    def get_initial_q(self):
        """Return the identifier of the DFA pre-trace state."""
        return self.initial_state

    def is_goal_reached(self, current_q):
        """Return whether the current DFA state is accepting."""
        return current_q in self.accepting_states

    def get_next_q(self, current_q, truth_assignment):
        """
        Evaluate outgoing transition guards and return the next DFA state.
        """
        if current_q not in self.transitions:
            return current_q
            
        for guard, next_q in self.transitions[current_q]:
            if self._eval_guard(guard, truth_assignment):
                return next_q
                
        return current_q

    def _eval_guard(self, guard, truth_assignment):
        """
        Convert a DOT guard such as "wp1 & ~wp2" to Python syntax and evaluate
        it against the current truth assignment.
        """
        guard = guard.strip()
        
        # Handle numeric and textual Boolean constants.
        if guard.lower() in ["1", "true"]: return True
        if guard.lower() in ["0", "false"]: return False
        
        # Convert the standard Boolean operators to Python syntax.
        expr = guard.replace('&', ' and ').replace('|', ' or ').replace('~', ' not ').replace('!', ' not ')
        
        try:
            # Disable built-ins while evaluating the Boolean expression.
            return eval(expr, {"__builtins__": {}}, truth_assignment)
        except Exception as e:
            print(f"[LTLfAutomaton error] Could not evaluate transition guard '{guard}': {e}")
            return False

    def render_graph(self, filename="ltlf_automaton", directory="img"):
        """Render the DFA and save it as a PNG image."""
        try:
            # ltlf2dfa emits a left-to-right graph.  With complex formulae the
            # transition guards become wide, leaving the resulting PNG only a
            # few pixels high.  A top-to-bottom layout gives labels enough room
            # and keeps the automaton readable independently of formula length.
            render_dot = re.sub(
                r"rankdir\s*=\s*LR\s*;",
                "rankdir = TB;",
                self.dot_string,
                count=1,
            )
            render_dot = re.sub(
                r"(digraph[^{]*\{)",
                (
                    r"\1\n"
                    r'graph [pad="0.35", nodesep="0.55", ranksep="0.75"];' "\n"
                    r'node [width="0.55", height="0.55"];' "\n"
                    r'edge [fontsize="10"];'
                ),
                render_dot,
                count=1,
            )
            src = Source(render_dot)
            src.render(filename=filename, directory=directory, format='png', cleanup=True)
            print(f"Automaton graph saved to: {directory}/{filename}.png")
        except Exception as e:
            print(f"[Graphviz error] Could not render the automaton graph: {e}")


class LTLfWaypointMDP:
    """
    Abstract MDP guided by an LTLf automaton.
    Each abstract state is (x, y, q), where q is the DFA state identifier.
    """
    def __init__(self, waypoints_dict, ltlf_automaton, width=12, height=12, gamma=0.99, goal_reward=10000):
        self.width = width
        self.height = height
        self.gamma = gamma
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7] # Include diagonal movements.
        
        self.waypoints_dict = waypoints_dict
        self.automaton = ltlf_automaton
        self.num_phases = self.automaton.num_phases
        
        # Generate every combination of grid position and DFA state.
        self.states = [(x, y, q) for x in range(width) for y in range(height) for q in self.automaton.states]
        
        self.goal_reward = goal_reward
        self.v_star = defaultdict(float)
        
    def _get_truth_assignment(self, x, y):
        """
        Map the current grid coordinates to a Boolean proposition assignment.
        """
        truth_assignment = {}
        for prop_name, (wp_x, wp_y) in self.waypoints_dict.items():
            truth_assignment[prop_name] = (x == wp_x and y == wp_y)
        return truth_assignment

    def get_transitions(self, state, action):
        x, y, q = state
        reward = 0
        
        # Apply the abstract physical movement.
        next_y = y
        if action in [0, 4, 5]:    next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:  next_y = max(y - 1, 0)
            
        next_x = x
        if action in [2, 4, 6]:    next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:  next_x = min(x + 1, self.width - 1)
        
        # Evaluate propositions at the arrival coordinates.
        truth_assignment = self._get_truth_assignment(next_x, next_y)
        
        # Advance the automaton using the arrival-state valuation.
        next_q = self.automaton.get_next_q(q, truth_assignment)

        next_state = (next_x, next_y, next_q)
        return next_state, reward

    def print_policy(self):
        arrows = {
            0: "↑",
            1: "↓",
            2: "←",
            3: "→",
            4: "↖",
            5: "↗",
            6: "↙",
            7: "↘"
        }

        for q in self.automaton.states:
            print(f"\n===== POLICY - DFA STATE q={q} =====")

            for y in reversed(range(self.height)):
                row = []

                for x in range(self.width):
                    state = (x, y, q)

                    if self.automaton.is_goal_reached(q):
                        row.append(" G ")
                        continue

                    best_action = None
                    best_value = -float("inf")

                    for a in self.actions:
                        next_state, reward = self.get_transitions(state, a)
                        value = reward + self.gamma * self.v_star[next_state]

                        if value > best_value:
                            best_value = value
                            best_action = a

                    row.append(f" {arrows[best_action]} ")

                print("".join(row))
    
    def value_iteration(self, theta=0.001):
        print(f"Value Iteration...")
        
        for s in self.states:
            if self.automaton.is_goal_reached(s[2]):
                self.v_star[s] = self.goal_reward
        
        while True:
            delta = 0
            new_v = self.v_star.copy()
            for s in self.states:
                if not self.automaton.is_goal_reached(s[2]):
                    v_actions = [self.get_transitions(s, a)[1] + self.gamma * self.v_star[self.get_transitions(s, a)[0]] for a in self.actions]
                    best_v = max(v_actions)
                    delta = max(delta, abs(best_v - self.v_star[s]))
                    new_v[s] = best_v
            self.v_star = new_v
            if delta < theta: break

        self.print_policy()
        


## 4. Write the Gymnasium action and LTLf task wrappers


In [ ]:
%%writefile agent.py
"""Gymnasium wrappers used by the SB3 SAC experiment."""

from collections import Counter

import gymnasium as gym
import numpy as np

from utils import phi_mapping_sequential


class DiscreteToContinuousActionWrapper(gym.ActionWrapper):
    """Expose a one-dimensional continuous action to a continuous-control agent.

    LunarLander still receives one of its original discrete actions.  A value
    selected by SAC is clipped to ``[-1, 1]``, scaled to ``[0, n)`` and floored.
    Clipping the final index is important because the upper Box endpoint maps
    exactly to ``n`` before flooring.
    """

    def __init__(self, env):
        super().__init__(env)
        if not isinstance(env.action_space, gym.spaces.Discrete):
            raise TypeError("The wrapped environment must have a Discrete action space")

        self.discrete_action_space = env.action_space
        self.action_space = gym.spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32,
        )

    def action(self, action):
        """Convert SAC's continuous action into an original LunarLander action."""
        continuous_action = np.asarray(action, dtype=np.float32)
        if continuous_action.size != 1:
            raise ValueError(
                "Expected one continuous action value, "
                f"received shape {continuous_action.shape}"
            )

        scalar_action = float(continuous_action.reshape(-1)[0])
        if not np.isfinite(scalar_action):
            raise ValueError("The continuous action must be finite")

        clipped_action = float(np.clip(scalar_action, -1.0, 1.0))
        unit_action = (clipped_action + 1.0) / 2.0
        discrete_action = int(np.floor(unit_action * self.discrete_action_space.n))
        return int(np.clip(discrete_action, 0, self.discrete_action_space.n - 1))


class LTLfTaskWrapper(gym.Wrapper):
    """Add the DFA state, synthetic task reward and potential-based shaping.

    This wrapper contains only environment semantics.  SAC construction,
    optimization, callbacks and checkpointing intentionally remain in
    ``trainer.py``.
    """

    def __init__(
        self,
        env,
        abstract_mdp,
        use_shaping=True,
        shaping_scale=1.0,
        goal_reward=10000.0,
        training_shaping_gamma=True,
    ):
        super().__init__(env)
        self.abstract_mdp = abstract_mdp
        self.automaton = abstract_mdp.automaton
        self.use_shaping = bool(use_shaping)
        self.shaping_scale = float(shaping_scale)
        self.goal_reward = float(goal_reward)
        self.training_shaping_gamma = bool(training_shaping_gamma)
        self.automaton_states = list(self.automaton.states)
        self.state_to_index = {
            q: index for index, q in enumerate(self.automaton_states)
        }

        if not isinstance(env.observation_space, gym.spaces.Box):
            raise TypeError("The wrapped environment must have a Box observation space")

        one_hot_low = np.zeros(len(self.automaton_states), dtype=np.float32)
        one_hot_high = np.ones(len(self.automaton_states), dtype=np.float32)
        self.observation_space = gym.spaces.Box(
            low=np.concatenate(
                (env.observation_space.low.astype(np.float32), one_hot_low)
            ),
            high=np.concatenate(
                (env.observation_space.high.astype(np.float32), one_hot_high)
            ),
            dtype=np.float32,
        )

        self.episode_metrics = []
        self.current_q = None
        self.raw_observation = None
        self._initially_accepted = False
        self._episode = None

    def _abstract_position(self, observation):
        """Map a raw LunarLander observation to abstract grid coordinates."""
        x, y, _ = phi_mapping_sequential(
            observation,
            0,
            self.abstract_mdp.width,
            self.abstract_mdp.height,
        )
        return x, y

    def _augment_observation(self, observation, q):
        """Append a one-hot encoding of the active DFA state."""
        if q not in self.state_to_index:
            raise RuntimeError(f"DFA returned unknown state {q!r}")
        one_hot = np.zeros(len(self.automaton_states), dtype=np.float32)
        one_hot[self.state_to_index[q]] = 1.0
        return np.concatenate((observation, one_hot)).astype(np.float32)

    def _initial_q(self, observation):
        """Consume the initial observation before the first agent action."""
        x, y = self._abstract_position(observation)
        truth_assignment = self.abstract_mdp._get_truth_assignment(x, y)
        pre_trace_q = self.automaton.get_initial_q()
        return self.automaton.get_next_q(pre_trace_q, truth_assignment)

    def _new_episode_metrics(self, q):
        """Create the mutable counters for one episode."""
        state_visits = [0] * len(self.automaton_states)
        state_entries = [0] * len(self.automaton_states)
        state_visits[self.state_to_index[q]] = 1
        state_entries[self.state_to_index[q]] = 1
        return {
            "task_reward": 0.0,
            "shaping_reward": 0.0,
            "learning_reward": 0.0,
            "episode_length": 0,
            "success": False,
            "initial_acceptance": False,
            "abstract_changes": 0,
            "dfa_transitions": 0,
            "transition_counter": Counter(),
            "state_visits": state_visits,
            "state_entries": state_entries,
            "env_terminated": False,
            "env_truncated": False,
        }

    def reset(self, **kwargs):
        """Reset LunarLander and initialize the DFA from the first observation."""
        observation, info = self.env.reset(**kwargs)
        self.raw_observation = observation
        self.current_q = self._initial_q(observation)
        self._episode = self._new_episode_metrics(self.current_q)
        self._initially_accepted = self.automaton.is_goal_reached(self.current_q)
        return self._augment_observation(observation, self.current_q), info

    def _finish_episode(self, info):
        """Freeze episode metrics and expose a compact copy through ``info``."""
        completed = self._episode.copy()
        completed["transition_counter"] = Counter(
            self._episode["transition_counter"]
        )
        completed["state_visits"] = list(self._episode["state_visits"])
        completed["state_entries"] = list(self._episode["state_entries"])
        self.episode_metrics.append(completed)
        info["ltlf_episode"] = {
            key: value
            for key, value in completed.items()
            if key not in {"transition_counter", "state_visits", "state_entries"}
        }

    def step(self, action):
        """Advance LunarLander and replace its reward with the LTLf task reward."""
        if self.current_q is None or self._episode is None:
            raise RuntimeError("reset() must be called before step()")

        # Gymnasium cannot mark a reset observation as terminal.  If s0 already
        # satisfies the task, emit one synthetic terminal transition without
        # applying the proposed action to LunarLander.
        if self._initially_accepted:
            self._initially_accepted = False
            self._episode["task_reward"] = self.goal_reward
            self._episode["learning_reward"] = self.goal_reward
            self._episode["success"] = True
            self._episode["initial_acceptance"] = True
            info = {"environment_reward": 0.0, "is_success": True}
            self._finish_episode(info)
            return (
                self._augment_observation(self.raw_observation, self.current_q),
                self.goal_reward,
                True,
                False,
                info,
            )

        previous_observation = self.raw_observation
        previous_q = self.current_q
        next_observation, environment_reward, env_terminated, env_truncated, info = (
            self.env.step(action)
        )
        info = dict(info)
        info["environment_reward"] = float(environment_reward)

        x, y = self._abstract_position(previous_observation)
        next_x, next_y = self._abstract_position(next_observation)
        abstract_state = (x, y, previous_q)

        truth_assignment = self.abstract_mdp._get_truth_assignment(next_x, next_y)
        next_q = self.automaton.get_next_q(previous_q, truth_assignment)
        if next_q not in self.state_to_index:
            raise RuntimeError(f"DFA returned unknown state {next_q!r}")
        abstract_next_state = (next_x, next_y, next_q)

        self._episode["state_visits"][self.state_to_index[next_q]] += 1
        if abstract_state != abstract_next_state:
            self._episode["abstract_changes"] += 1

        if next_q != previous_q:
            self._episode["dfa_transitions"] += 1
            self._episode["state_entries"][self.state_to_index[next_q]] += 1
            self._episode["transition_counter"][(previous_q, next_q)] += 1

        success = self.automaton.is_goal_reached(next_q)
        task_reward = self.goal_reward if success else 0.0
        shaping_reward = 0.0
        if self.use_shaping and abstract_state != abstract_next_state:
            phi_state = self.abstract_mdp.v_star.get(abstract_state, 0.0)
            phi_next_state = self.abstract_mdp.v_star.get(abstract_next_state, 0.0)
            training_discount = (
                self.abstract_mdp.gamma if self.training_shaping_gamma else 1.0
            )
            shaping_reward = self.shaping_scale * (
                training_discount * phi_next_state - phi_state
            )
        learning_reward = task_reward + shaping_reward

        self._episode["task_reward"] += task_reward
        self._episode["shaping_reward"] += shaping_reward
        self._episode["learning_reward"] += learning_reward
        self._episode["episode_length"] += 1
        self._episode["success"] = success
        self._episode["env_terminated"] = bool(env_terminated)
        self._episode["env_truncated"] = bool(env_truncated)

        self.raw_observation = next_observation
        self.current_q = next_q
        terminated = bool(env_terminated or success)
        truncated = bool(env_truncated)
        info["is_success"] = success

        if terminated or truncated:
            self._finish_episode(info)

        return (
            self._augment_observation(next_observation, next_q),
            float(learning_reward),
            terminated,
            truncated,
            info,
        )


## 5. Write plotting and abstraction utilities


In [ ]:
%%writefile utils.py
"""Spatial mapping and plotting utilities for the SAC framework."""

# ==============================
# Standard library imports
# ==============================

import os

# ==============================
# External imports
# ==============================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle


# ==============================
# Spatial discretization and grid geometry
# ==============================

# Legacy discretization. To reactivate it, uncomment this function and comment
# out the active phi_mapping_grid implementation immediately below.
#
# def phi_mapping_grid(obs, grid_w=12, grid_h=12):
#     """Map coordinates using the original grid_size - 1 discretization."""
#     x, y = float(obs[0]), float(obs[1])
#     abstract_x = int(np.clip((x + 1.0) / 2.0 * (grid_w - 1), 0, grid_w - 1))
#     abstract_y = int(np.clip(y / 1.5 * (grid_h - 1), 0, grid_h - 1))
#     return abstract_x, abstract_y


def phi_mapping_grid(obs, grid_w=12, grid_h=12):
    """Map LunarLander coordinates to uniform bins over x=[-1,1], y=[0,1.5]."""
    if grid_w <= 0 or grid_h <= 0:
        raise ValueError("grid_w and grid_h must be positive")

    x, y = float(obs[0]), float(obs[1])
    abstract_x = int(np.floor((x + 1.0) / 2.0 * grid_w))
    abstract_y = int(np.floor(y / 1.5 * grid_h))
    abstract_x = int(np.clip(abstract_x, 0, grid_w - 1))
    abstract_y = int(np.clip(abstract_y, 0, grid_h - 1))
    return abstract_x, abstract_y


def _axis_boundaries(map_axis, size, lower, upper):
    """Infer bin boundaries from the active mapper."""
    if size <= 0:
        raise ValueError("grid dimensions must be positive")

    boundaries = [float(lower)]
    iterations = 60
    for target_index in range(1, size):
        left, right = float(lower), float(upper)
        for _ in range(iterations):
            midpoint = (left + right) / 2.0
            if map_axis(midpoint) < target_index:
                left = midpoint
            else:
                right = midpoint
        boundaries.append(right)
    boundaries.append(float(upper))
    return np.asarray(boundaries, dtype=float)


def spatial_grid_boundaries(grid_w=12, grid_h=12):
    """Return x/y bin boundaries implied by the active phi_mapping_grid."""
    x_boundaries = _axis_boundaries(
        lambda x: phi_mapping_grid((x, 0.0), grid_w, grid_h)[0],
        grid_w,
        -1.0,
        1.0,
    )
    y_boundaries = _axis_boundaries(
        lambda y: phi_mapping_grid((0.0, y), grid_w, grid_h)[1],
        grid_h,
        0.0,
        1.5,
    )
    return x_boundaries, y_boundaries


def phi_mapping_sequential(obs, q, grid_w=12, grid_h=12):
    abstract_x, abstract_y = phi_mapping_grid(obs, grid_w, grid_h)
    return abstract_x, abstract_y, q


def lunar_lander_visible_observation_bounds():
    """Return the normalised x/y bounds covered by LunarLander's RGB viewport."""
    from gymnasium.envs.box2d import lunar_lander

    viewport_world_width = lunar_lander.VIEWPORT_W / lunar_lander.SCALE
    viewport_world_height = lunar_lander.VIEWPORT_H / lunar_lander.SCALE
    helipad_y = viewport_world_height / 4.0
    lander_y_offset = helipad_y + lunar_lander.LEG_DOWN / lunar_lander.SCALE
    half_world_height = viewport_world_height / 2.0
    visible_y_min = (0.0 - lander_y_offset) / half_world_height
    visible_y_max = (viewport_world_height - lander_y_offset) / half_world_height
    return -1.0, 1.0, visible_y_min, visible_y_max


def _draw_visible_area_overlay(axis, width, height):
    """Mark which portion of the active abstract grid lies in the RGB viewport."""
    visible_x_min, visible_x_max, visible_y_min, visible_y_max = (
        lunar_lander_visible_observation_bounds()
    )
    x_boundaries, y_boundaries = spatial_grid_boundaries(width, height)

    def to_plot(value, boundaries):
        value = float(np.clip(value, boundaries[0], boundaries[-1]))
        index = int(np.searchsorted(boundaries, value, side="right") - 1)
        index = int(np.clip(index, 0, len(boundaries) - 2))
        lower, upper = boundaries[index], boundaries[index + 1]
        fraction = 0.0 if upper <= lower else (value - lower) / (upper - lower)
        return index - 0.5 + fraction

    left = float(to_plot(visible_x_min, x_boundaries))
    right = float(to_plot(visible_x_max, x_boundaries))
    bottom = float(to_plot(visible_y_min, y_boundaries))
    top = float(to_plot(visible_y_max, y_boundaries))

    axis.add_patch(
        Rectangle(
            (left, bottom),
            right - left,
            top - bottom,
            fill=False,
            edgecolor="#ff1744",
            linewidth=1.4,
            linestyle="--",
            label="Visible RGB viewport",
            zorder=5,
        )
    )
    axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

# ==============================
# Abstract-potential heatmaps
# ==============================


def save_sequential_heatmaps(
    abstract_mdp,
    filename_prefix="v_star",
    output_dir=None,
):
    """
    Generates and saves a separate heatmap for V* for each phase defined in the MDP,
    without any waypoint or goal markers (clean heatmap).
    """
    output_dir = output_dir or os.path.join("img", "heatmaps")
    os.makedirs(output_dir, exist_ok=True)
    filename_prefix = os.path.basename(filename_prefix)
    
    width, height = abstract_mdp.width, abstract_mdp.height
    
    # Extract global min/max for consistent colormap scaling
    all_values = np.array(list(abstract_mdp.v_star.values()))
    computed_vmin = all_values.min() if len(all_values) > 0 else 0
    computed_vmax = all_values.max() if len(all_values) > 0 else 1

    for current_q in abstract_mdp.automaton.states:
        matrix = np.zeros((height, width))
        for (x, y, q), value in abstract_mdp.v_star.items():
            if q == current_q and 0 <= x < width and 0 <= y < height:
                matrix[y, x] = value
                
        plt.figure(figsize=(9, 8))
        im = plt.imshow(matrix, cmap='viridis', origin='lower', vmin=computed_vmin, vmax=computed_vmax)
        
        for y in range(height):
            for x in range(width):
                val = matrix[y, x]
                if val > 0.0: 
                    text_color = 'white' if val < (computed_vmax / 2) else 'black'
                    plt.text(x, y, f"{val:.1f}", ha='center', va='center', color=text_color, fontsize=7)
                    
        plt.colorbar(im, fraction=0.046, pad=0.04, label="Potential Value (V*)")
        
        is_goal_state = abstract_mdp.automaton.is_goal_reached(current_q)
        phase_label = "Goal Reached" if is_goal_state else "Seeking Targets"
        plt.title(f"Potential Map (V*) - DFA State q={current_q} ({phase_label})", fontsize=14, fontweight='bold')
        
        ax = plt.gca()
        ax.set_xticks(np.arange(-.5, width, 1), minor=True)
        ax.set_yticks(np.arange(-.5, height, 1), minor=True)
        ax.grid(which='minor', color='w', linestyle='-', linewidth=1, alpha=0.4)
        _draw_visible_area_overlay(ax, width, height)
        
        # Keep the heatmap free of waypoint and goal markers.
            
        plt.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
        plt.savefig(os.path.join(output_dir, f"{filename_prefix}_q{current_q}.png"), dpi=150, bbox_inches='tight')
        plt.close()
        print(f" -> Generated V* Heatmap for DFA State q={current_q}")

# ==============================
# Training diagnostics and learning curves
# ==============================


def plot_training_variance(reward_histories, window_size=100, title="Training Performance Across Seeds", filename="img/training_variance.png", label="Learning reward"):
    """Plot the smoothed mean reward and its variance across training seeds."""
    runs = np.asarray(reward_histories, dtype=np.float64)
    if runs.ndim == 1:
        runs = runs[np.newaxis, :]
    if runs.ndim != 2 or runs.shape[0] == 0 or runs.shape[1] == 0:
        raise ValueError("reward_histories must have shape (num_seeds, episodes)")
    if window_size <= 0:
        raise ValueError("window_size must be greater than zero")

    smoothed_runs = (
        pd.DataFrame(runs.T)
        .rolling(window=window_size, min_periods=1, center=False)
        .mean()
        .to_numpy()
        .T
    )
    mean_reward = np.mean(smoothed_runs, axis=0)
    variance = np.var(smoothed_runs, axis=0)
    std_reward = np.sqrt(variance)
    episodes = np.arange(1, runs.shape[1] + 1)

    output_directory = os.path.dirname(os.fspath(filename))
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(episodes, mean_reward, color="tab:blue", linewidth=2.5, label=f"Mean {label}")
    ax.fill_between(
        episodes,
        mean_reward - std_reward,
        mean_reward + std_reward,
        color="tab:blue",
        alpha=0.2,
        label="±1 std across seeds (variance = σ²)",
    )
    ax.set_title(f"{title} ({runs.shape[0]} seeds)", fontsize=15, fontweight="bold")
    ax.set_xlabel(f"Episode (mean over the last {window_size} episodes)", fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="best", fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches="tight")
    print(f"\n>>> Training variance plot saved to: {filename}")
    plt.close(fig)

def plot_buffer_fractions(buffer_histories, window_size=100, filename="img/buffer_fractions.png", state_labels=None):
    """
    Plots the replay buffer composition for N phases dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    x_axis = np.arange(len(buffer_histories[0]))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(buffer_histories)))
    for idx, history in enumerate(buffer_histories):
        ma = pd.Series(history).rolling(window=window_size, min_periods=1, center=False).mean()
        state_label = state_labels[idx] if state_labels is not None else idx
        ax.plot(x_axis, ma, color=colors[idx], linewidth=2.5, label=f'DFA state q={state_label}')
    
    ax.set_title(f"Replay Buffer Composition (MA Window = {window_size})", fontsize=14, fontweight='bold')
    ax.set_ylabel("Fraction in Buffer", fontsize=12)
    ax.set_ylim(0, 1.05)
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(buffer_histories), fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)

def plot_shaping_reward_breakdown(true_rewards, total_rewards, exploration_history, window_size=100, filename="img/shaping_reward_breakdown.png", exploration_label="Entropy coefficient (alpha)"):
    """
    Plot reward moving averages and SAC's entropy coefficient.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Moving Average Calculation
    true_ma = pd.Series(true_rewards).rolling(window=window_size, min_periods=1, center=False).mean()
    total_ma = pd.Series(total_rewards).rolling(window=window_size, min_periods=1, center=False).mean()
        
    x_axis = np.arange(len(true_rewards))
        
    # Plot Rewards (Left Y-Axis)
    ax1.plot(x_axis, true_ma, color='green', linestyle='-', linewidth=2, label='Synthetic Goal Reward')
    ax1.plot(x_axis, total_ma, color='purple', linestyle='-', linewidth=2.5, label='Learning Reward (Goal + Shaping)')
    
    ax1.set_title(f"SAC Reward Analysis (MA Window = {window_size})", fontsize=15, fontweight='bold')
    ax1.set_xlabel("Episode #", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # SAC does not use epsilon-greedy exploration.  Its learned entropy
    # coefficient is the corresponding exploration diagnostic.
    ax2 = ax1.twinx()
    ax2.plot(
        x_axis,
        exploration_history,
        color='orange',
        linestyle='--',
        linewidth=1.8,
        label=exploration_label,
    )
    ax2.set_ylabel(exploration_label, color='black', fontsize=12)

    # Combine Legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Dynamically calculate legend columns based on number of items to keep it compact
    legend_cols = max(2, (len(labels1) + len(labels2)) // 2)
    
    ax1.legend(
        lines1 + lines2, labels1 + labels2, 
        loc="upper center", bbox_to_anchor=(0.5, -0.15), 
        ncol=legend_cols, fontsize=11, framealpha=1.0
    )
    
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)


## 6. Write the automaton validator


In [ ]:
%%writefile automaton_validator.py
# ==============================
# Standard library imports
# ==============================

import itertools
import re
from collections import deque


# ==============================
# Formula and valuation helpers
# ==============================

LTLF_OPERATORS = {"F", "G", "M", "R", "U", "W", "X", "false", "true"}


def _extract_formula_propositions(formula):
    """Extract atomic proposition names from an LTLf formula."""
    tokens = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", formula))
    return sorted(token for token in tokens if token not in LTLF_OPERATORS)


def _generate_truth_assignments(propositions):
    """Generate every Boolean valuation for the formula propositions."""
    for values in itertools.product((False, True), repeat=len(propositions)):
        yield dict(zip(propositions, values))


def _matching_transitions(automaton, state, truth_assignment):
    """Return all outgoing DFA transitions enabled by one truth assignment."""
    return [(guard, destination) for guard, destination in automaton.transitions.get(state, []) if automaton._eval_guard(guard, truth_assignment)]


# ==============================
# Validation report
# ==============================

class AutomatonValidationReport:
    """Collect validation errors, warnings, and useful DFA statistics."""

    def __init__(self, formula, propositions):
        self.formula = formula
        self.propositions = propositions
        self.errors = []
        self.warnings = []
        self.statistics = {}

    @property
    def is_valid(self):
        """Return whether validation completed without errors."""
        return not self.errors

    def add_error(self, message):
        """Append one blocking validation error."""
        self.errors.append(message)

    def add_warning(self, message):
        """Append one non-blocking validation warning."""
        self.warnings.append(message)

    def format(self):
        """Format the complete validation report for console and log output."""
        status = "VALID" if self.is_valid else "INVALID"
        lines = [
            "=== AUTOMATON VALIDATION ===",
            f"Status: {status}",
            f"Formula propositions: {self.propositions}",
            f"Statistics: {self.statistics}",
        ]
        if self.errors:
            lines.append("Errors:")
            lines.extend(f"- {message}" for message in self.errors)
        if self.warnings:
            lines.append("Warnings:")
            lines.extend(f"- {message}" for message in self.warnings)
        return "\n".join(lines)

    def raise_if_invalid(self):
        """Raise an exception containing the report when validation fails."""
        if not self.is_valid:
            raise ValueError(self.format())


# ==============================
# DFA validation
# ==============================

def validate_automaton(automaton, waypoints_dict, width=None, height=None, max_propositions=12, raise_on_error=True):
    """Validate DFA structure, guards, reachability, propositions, and waypoint coordinates."""
    propositions = _extract_formula_propositions(automaton.formula_str)
    report = AutomatonValidationReport(automaton.formula_str, propositions)
    states = set(automaton.states)
    waypoint_propositions = set(waypoints_dict)

    # Validate formula propositions and waypoint declarations.
    missing_waypoints = sorted(set(propositions) - waypoint_propositions)
    unused_waypoints = sorted(waypoint_propositions - set(propositions))
    if missing_waypoints:
        report.add_error(f"Formula propositions without coordinates: {missing_waypoints}")
    if unused_waypoints:
        report.add_warning(f"Waypoint propositions not used by the formula: {unused_waypoints}")
    if len(propositions) > max_propositions:
        report.add_error(f"The formula has {len(propositions)} propositions; exhaustive validation is limited to {max_propositions}")

    # Validate waypoint coordinate structure and grid bounds.
    for proposition, coordinates in waypoints_dict.items():
        if not isinstance(coordinates, (tuple, list)) or len(coordinates) != 2:
            report.add_error(f"Waypoint {proposition!r} must contain exactly two coordinates")
            continue
        x, y = coordinates
        if not isinstance(x, int) or not isinstance(y, int):
            report.add_error(f"Waypoint {proposition!r} coordinates must be integers")
        if width is not None and not 0 <= x < width:
            report.add_error(f"Waypoint {proposition!r} has x={x}, outside [0, {width - 1}]")
        if height is not None and not 0 <= y < height:
            report.add_error(f"Waypoint {proposition!r} has y={y}, outside [0, {height - 1}]")

    # Validate initial, accepting, source, and destination states.
    if not states:
        report.add_error("The DFA contains no states")
    if automaton.get_initial_q() not in states:
        report.add_error(f"Initial state {automaton.get_initial_q()!r} is not part of the DFA")
    unknown_accepting = sorted(set(automaton.accepting_states) - states)
    if unknown_accepting:
        report.add_error(f"Unknown accepting states: {unknown_accepting}")
    if not automaton.accepting_states:
        report.add_error("The DFA has no accepting states")
    for source, transitions in automaton.transitions.items():
        if source not in states:
            report.add_error(f"Transition source {source!r} is not part of the DFA")
        for guard, destination in transitions:
            if destination not in states:
                report.add_error(f"Transition {source!r} --[{guard}]--> {destination!r} targets an unknown state")

    # Validate deterministic and complete behavior over the full Boolean alphabet.
    truth_assignments = list(_generate_truth_assignments(propositions)) if len(propositions) <= max_propositions else []
    reachable_states = {automaton.get_initial_q()} if automaton.get_initial_q() in states else set()
    frontier = deque(reachable_states)
    checked_pairs = 0
    ambiguous_pairs = 0
    incomplete_pairs = 0

    for state in states:
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            checked_pairs += 1
            if not matches:
                incomplete_pairs += 1
                report.add_error(f"No transition from state {state!r} for valuation {truth_assignment}")
            elif len(matches) > 1:
                ambiguous_pairs += 1
                guards = [guard for guard, _ in matches]
                report.add_error(f"Ambiguous transitions from state {state!r} for valuation {truth_assignment}: {guards}")
            else:
                expected_destination = matches[0][1]
                actual_destination = automaton.get_next_q(state, truth_assignment)
                if actual_destination != expected_destination:
                    report.add_error(f"get_next_q returned {actual_destination!r}, expected {expected_destination!r} from state {state!r}")

    # Traverse the validated transition relation to find unreachable states.
    while frontier and truth_assignments:
        state = frontier.popleft()
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            if len(matches) == 1 and matches[0][1] not in reachable_states:
                reachable_states.add(matches[0][1])
                frontier.append(matches[0][1])

    unreachable_states = sorted(states - reachable_states)
    unreachable_accepting = sorted(set(automaton.accepting_states) - reachable_states)
    if unreachable_states:
        report.add_warning(f"Unreachable DFA states: {unreachable_states}")
    if unreachable_accepting:
        report.add_error(f"No accepting path reaches states: {unreachable_accepting}")

    # Store compact statistics for diagnostics and reproducibility.
    report.statistics = {
        "states": len(states),
        "accepting_states": len(automaton.accepting_states),
        "transitions": sum(len(transitions) for transitions in automaton.transitions.values()),
        "valuations": len(truth_assignments),
        "state_valuation_pairs": checked_pairs,
        "ambiguous_pairs": ambiguous_pairs,
        "incomplete_pairs": incomplete_pairs,
        "reachable_states": len(reachable_states),
    }

    if raise_on_error:
        report.raise_if_invalid()
    return report


## 7. Write the SB3 SAC training program


In [ ]:
%%writefile trainer.py
"""Train Stable-Baselines3 SAC on the LTLf LunarLander task."""

# ==============================
# Standard library imports
# ==============================

import argparse
import json
import re
from collections import Counter
from pathlib import Path

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback

from abstract_mdps import LTLfAutomaton, LTLfWaypointMDP
from agent import DiscreteToContinuousActionWrapper, LTLfTaskWrapper
from automaton_validator import validate_automaton
from utils import (
    plot_buffer_fractions,
    plot_shaping_reward_breakdown,
    plot_training_variance,
    save_sequential_heatmaps,
)


SCRIPT_DIR = Path(__file__).resolve().parent


# ==============================
# Data helpers
# ==============================

def _positive_int(value):
    """Parse a strictly positive command-line integer."""
    number = int(value)
    if number <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return number


def _experiment_name(value):
    """Validate a safe single-directory experiment name."""
    name = str(value).strip()
    if len(name) > 100 or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", name):
        raise argparse.ArgumentTypeError(
            "must start with a letter or digit and contain only letters, digits, '.', '_' or '-'"
        )
    return name


def _parse_entropy_coefficient(value):
    """Accept SB3's automatic modes or a positive fixed coefficient."""
    text = str(value).strip()
    if text == "auto" or text.startswith("auto_"):
        if text != "auto":
            try:
                initial_value = float(text.removeprefix("auto_"))
            except ValueError as error:
                raise argparse.ArgumentTypeError(
                    "must be 'auto', 'auto_<initial_value>' or a positive number"
                ) from error
            if initial_value <= 0:
                raise argparse.ArgumentTypeError(
                    "the automatic entropy coefficient must start above zero"
                )
        return text

    try:
        coefficient = float(text)
    except ValueError as error:
        raise argparse.ArgumentTypeError(
            "must be 'auto', 'auto_<initial_value>' or a positive number"
        ) from error
    if coefficient <= 0:
        raise argparse.ArgumentTypeError(
            "the entropy coefficient must be greater than zero"
        )
    return coefficient

def save_training_data(filename, **kwargs):
    """Convert training metrics to arrays and save them in a compressed NPZ."""
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    np_data = {key: np.asarray(value) for key, value in kwargs.items()}
    if any(array.dtype == object for array in np_data.values()):
        raise ValueError("Training metrics must be rectangular numeric arrays")
    np.savez_compressed(filename, **np_data)
    print(f"\nTraining data saved to: {filename}")


def _aggregate_seed_metrics(seed_metrics, seeds):
    """Keep first-run compatibility and add every metric stacked by seed."""
    if not seed_metrics:
        raise ValueError("At least one seed run is required")
    aggregated = dict(seed_metrics[0])
    aggregated["seeds"] = np.asarray(seeds, dtype=np.int64)
    for key in seed_metrics[0]:
        if key == "automaton_states":
            continue
        try:
            aggregated[f"{key}_runs"] = np.stack(
                [np.asarray(metrics[key]) for metrics in seed_metrics]
            )
        except ValueError as error:
            raise ValueError(f"Metric {key!r} has inconsistent shapes across seeds") from error
    for key in ("task_rewards", "learning_rewards", "shaping_rewards"):
        runs = aggregated[f"{key}_runs"]
        aggregated[f"{key}_mean"] = np.mean(runs, axis=0)
        aggregated[f"{key}_variance"] = np.var(runs, axis=0)
    return aggregated


def _format_counter(counter):
    """Convert a DFA transition counter into a compact readable string."""
    if not counter:
        return "none"
    return ", ".join(
        f"{source}->{destination}: {count}"
        for (source, destination), count in sorted(counter.items())
    )


def _replay_buffer_fractions(model, num_states):
    """Return the fraction of stored observations belonging to each DFA state."""
    replay_buffer = model.replay_buffer
    fractions = np.zeros(num_states, dtype=np.float64)
    if replay_buffer is None or replay_buffer.size() == 0:
        return fractions

    # SB3 stores observations as [buffer index, environment index, feature].
    valid_size = replay_buffer.buffer_size if replay_buffer.full else replay_buffer.pos
    observations = replay_buffer.observations[:valid_size]
    observations = observations.reshape(-1, observations.shape[-1])
    phase_features = observations[:, -num_states:]
    phase_indices = np.argmax(phase_features, axis=1)
    counts = np.bincount(phase_indices, minlength=num_states)
    return counts.astype(np.float64) / len(phase_indices)


def _entropy_coefficient(model):
    """Read SAC's current entropy coefficient for diagnostics."""
    if model.log_ent_coef is not None:
        return float(model.log_ent_coef.detach().exp().cpu().item())
    return float(model.ent_coef)


def _build_training_results(task_wrapper, callback):
    """Collect numeric episode histories under stable descriptive names."""
    metrics = task_wrapper.episode_metrics
    return {
        "task_rewards": [episode["task_reward"] for episode in metrics],
        "learning_rewards": [episode["learning_reward"] for episode in metrics],
        "shaping_rewards": [episode["shaping_reward"] for episode in metrics],
        "entropy_coefficient_history": callback.entropy_history,
        "buffer_histories": np.asarray(callback.buffer_histories).T,
        "state_visit_histories": np.asarray(
            [episode["state_visits"] for episode in metrics]
        ).T,
        "state_entry_histories": np.asarray(
            [episode["state_entries"] for episode in metrics]
        ).T,
        "successes": [int(episode["success"]) for episode in metrics],
        "initial_acceptances": [
            int(episode["initial_acceptance"]) for episode in metrics
        ],
        "episode_lengths": [episode["episode_length"] for episode in metrics],
        "abstract_changes": [episode["abstract_changes"] for episode in metrics],
        "dfa_transitions": [episode["dfa_transitions"] for episode in metrics],
        "automaton_states": task_wrapper.automaton_states,
        "best_mean_learning_reward": callback.best_mean_reward,
        "best_policy_episode": callback.best_policy_episode,
    }


# ==============================
# SAC callback and training
# ==============================

class SACTrainingCallback(BaseCallback):
    """Collect episode metrics, print diagnostics and stop at an exact count."""

    def __init__(self, task_wrapper, episodes, log_interval, policy_dir, save_policy=True, log_file=None):
        super().__init__(verbose=0)
        if episodes <= 0:
            raise ValueError("episodes must be greater than zero")
        if log_interval <= 0:
            raise ValueError("log_interval must be greater than zero")

        self.task_wrapper = task_wrapper
        self.episodes = int(episodes)
        self.log_interval = int(log_interval)
        self.policy_dir = Path(policy_dir)
        self.save_policy = bool(save_policy)
        self.log_file = Path(log_file) if log_file else None
        self.processed_episodes = 0
        self.buffer_histories = []
        self.entropy_history = []
        self.best_mean_reward = -np.inf
        self.best_policy_episode = 0
        self.cumulative_state_visits = Counter()
        self.cumulative_state_entries = Counter()
        self.cumulative_transitions = Counter()
        self.cumulative_initial_acceptances = 0
        self.cumulative_env_terminated = 0
        self.cumulative_env_truncated = 0
        self._log_handle = None

    def _write(self, message):
        """Print a message and append it to the run log when configured."""
        print(message)
        if self._log_handle:
            self._log_handle.write(message)
            self._log_handle.flush()

    def _on_training_start(self):
        """Open the log and record the immutable experiment configuration."""
        if self.log_file:
            self.log_file.parent.mkdir(parents=True, exist_ok=True)
            self._log_handle = self.log_file.open("a", encoding="utf-8")

        abstract_mdp = self.task_wrapper.abstract_mdp
        automaton = abstract_mdp.automaton
        self._write(
            "\n=== NEW SAC RUN ===\n"
            f"episodes={self.episodes}, shaping={self.task_wrapper.use_shaping}, "
            f"K={self.task_wrapper.shaping_scale}, "
            f"goal_reward={self.task_wrapper.goal_reward}, gamma={abstract_mdp.gamma}\n"
            f"training_shaping_gamma={self.task_wrapper.training_shaping_gamma}, "
            f"shaping_formula={'K*(gamma*Phi(next)-Phi(state))' if self.task_wrapper.training_shaping_gamma else 'K*(Phi(next)-Phi(state))'}\n"
            f"formula={automaton.formula_str}\n"
            f"waypoints={abstract_mdp.waypoints_dict}\n"
            f"dfa_states={self.task_wrapper.automaton_states}, "
            f"pre_trace={automaton.get_initial_q()}, "
            f"accepting={sorted(automaton.accepting_states)}\n"
        )

    def _update_cumulative_counters(self, episode):
        """Merge one completed episode into cumulative diagnostic counters."""
        for index, q in enumerate(self.task_wrapper.automaton_states):
            self.cumulative_state_visits[q] += episode["state_visits"][index]
            self.cumulative_state_entries[q] += episode["state_entries"][index]
        self.cumulative_transitions.update(episode["transition_counter"])
        self.cumulative_initial_acceptances += int(episode["initial_acceptance"])
        self.cumulative_env_terminated += int(episode["env_terminated"])
        self.cumulative_env_truncated += int(episode["env_truncated"])

    def _should_log(self):
        """Return whether the latest episode closes a monitoring window."""
        return (
            self.processed_episodes == 1
            or self.processed_episodes == self.episodes
            or self.processed_episodes % self.log_interval == 0
        )

    def _training_log(self):
        """Build a report from the latest monitoring window."""
        metrics = self.task_wrapper.episode_metrics[: self.processed_episodes]
        window = min(self.log_interval, self.processed_episodes)
        recent = metrics[-window:]
        recent_transitions = Counter()
        for episode in recent:
            recent_transitions.update(episode["transition_counter"])

        states = self.task_wrapper.automaton_states
        recent_visits = np.asarray(
            [episode["state_visits"] for episode in recent], dtype=np.int64
        ).sum(axis=0)
        recent_entries = np.asarray(
            [episode["state_entries"] for episode in recent], dtype=np.int64
        ).sum(axis=0)
        latest_fractions = self.buffer_histories[-1]

        def average(key):
            return float(np.mean([episode[key] for episode in recent]))

        return (
            "\n"
            f"[Episode {self.processed_episodes}/{self.episodes} | last {window}]\n"
            f"success rate                : {average('success'):.1%} "
            f"(cumulative {np.mean([e['success'] for e in metrics]):.1%})\n"
            f"synthetic task reward       : {average('task_reward'):.3f}\n"
            f"shaping reward              : {average('shaping_reward'):.3f}\n"
            f"learning reward             : {average('learning_reward'):.3f}\n"
            f"episode length              : {average('episode_length'):.1f}\n"
            f"abstract changes / episode  : {average('abstract_changes'):.1f}\n"
            f"DFA transitions / episode   : {average('dfa_transitions'):.2f}\n"
            f"DFA transitions in window   : {_format_counter(recent_transitions)}\n"
            f"entropy coefficient          : {self.entropy_history[-1]:.6f}\n"
            f"replay buffer                : {self.model.replay_buffer.size()} samples "
            f"[{', '.join(f'{q}: {latest_fractions[i]:.1%}' for i, q in enumerate(states))}]\n"
            f"DFA state visits in window   : "
            f"{', '.join(f'{q}: {recent_visits[i]}' for i, q in enumerate(states))}\n"
            f"DFA state visits cumulative  : "
            f"{', '.join(f'{q}: {self.cumulative_state_visits[q]}' for q in states)}\n"
            f"DFA state entries in window  : "
            f"{', '.join(f'{q}: {recent_entries[i]}' for i, q in enumerate(states))}\n"
            f"DFA state entries cumulative : "
            f"{', '.join(f'{q}: {self.cumulative_state_entries[q]}' for q in states)}\n"
            f"transitions cumulative       : {_format_counter(self.cumulative_transitions)}\n"
            f"accepted directly from s0    : {self.cumulative_initial_acceptances}\n"
            f"Gym endings cumulative       : terminated={self.cumulative_env_terminated}, "
            f"truncated={self.cumulative_env_truncated}\n"
        )

    def _save_best_policy(self):
        """Replace the best checkpoint when monitored learning reward improves."""
        metrics = self.task_wrapper.episode_metrics[: self.processed_episodes]
        window = min(self.log_interval, self.processed_episodes)
        monitored_mean = float(
            np.mean([episode["learning_reward"] for episode in metrics[-window:]])
        )
        if monitored_mean <= self.best_mean_reward:
            return

        self.best_mean_reward = monitored_mean
        self.best_policy_episode = self.processed_episodes
        if self.save_policy:
            self.policy_dir.mkdir(parents=True, exist_ok=True)
            self.model.save(self.policy_dir / "best_policy")
        self._write(
            f"Best policy updated at episode {self.best_policy_episode}: "
            f"mean learning reward={self.best_mean_reward:.3f}\n"
        )

    def _process_completed_episode(self, episode):
        """Record replay, entropy and task metrics for one completed episode."""
        self.processed_episodes += 1
        self._update_cumulative_counters(episode)
        self.buffer_histories.append(
            _replay_buffer_fractions(self.model, len(self.task_wrapper.automaton_states))
        )
        self.entropy_history.append(_entropy_coefficient(self.model))
        if self._should_log():
            self._write(self._training_log())
            self._save_best_policy()

    def _on_step(self):
        """Process newly completed episodes and stop at the requested total."""
        while self.processed_episodes < len(self.task_wrapper.episode_metrics):
            episode = self.task_wrapper.episode_metrics[self.processed_episodes]
            self._process_completed_episode(episode)
        return self.processed_episodes < self.episodes

    def _on_training_end(self):
        """Close the append-only log even when training stops via the callback."""
        if self._log_handle:
            self._log_handle.close()
            self._log_handle = None


def run_sequential_training(env, task_wrapper, abstract_mdp, episodes, policy_dir, log_file, log_interval=100, learning_rate=3e-4, buffer_size=300000, learning_starts=100, batch_size=256, tau=0.005, ent_coef="auto", seed=None, device="auto", save_policy=True):
    """Construct and train SB3 SAC; all optimization logic lives here."""
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        learning_starts=learning_starts,
        batch_size=batch_size,
        tau=tau,
        gamma=abstract_mdp.gamma,
        train_freq=(1, "step"),
        gradient_steps=1,
        ent_coef=ent_coef,
        policy_kwargs={"net_arch": [128, 128]},
        seed=seed,
        device=device,
        verbose=0,
    )
    print(f"Using device: {model.device}")

    callback = SACTrainingCallback(
        task_wrapper=task_wrapper,
        episodes=episodes,
        log_interval=log_interval,
        policy_dir=policy_dir,
        save_policy=save_policy,
        log_file=log_file,
    )

    max_episode_steps = getattr(env.spec, "max_episode_steps", None) or 1000
    model.learn(
        total_timesteps=int(episodes) * int(max_episode_steps),
        callback=callback,
        log_interval=log_interval,
    )

    if callback.processed_episodes != episodes:
        raise RuntimeError(
            f"Training stopped after {callback.processed_episodes} episodes; "
            f"expected {episodes}"
        )

    if save_policy:
        policy_dir = Path(policy_dir)
        policy_dir.mkdir(parents=True, exist_ok=True)
        model.save(policy_dir / "last_policy")
        print(
            f"Last policy saved after episode {episodes}. Best policy: episode "
            f"{callback.best_policy_episode}, mean learning reward="
            f"{callback.best_mean_reward:.3f}"
        )

    return _build_training_results(task_wrapper, callback)


# ==============================
# Experiment setup and outputs
# ==============================

def main(args):
    """Configure SAC training or regenerate plots from saved numeric metrics."""
    if args.num_seeds <= 0:
        raise ValueError("num_seeds must be greater than zero")
    experiment_dir = SCRIPT_DIR / "results" / args.experiment_name
    data_dir = experiment_dir
    image_dir = experiment_dir / "img"
    log_dir = experiment_dir / "logs"
    policy_dir = experiment_dir / "policy"
    for directory in (data_dir, image_dir, log_dir, policy_dir):
        directory.mkdir(parents=True, exist_ok=True)
    plot_dir = image_dir
    print(f"Experiment outputs: {experiment_dir}")

    with Path(args.config).expanduser().open(encoding="utf-8") as config_file:
        config = json.load(config_file)

    formula = config.get("formula", "F(goal)")
    waypoints = {
        name: tuple(coordinates)
        for name, coordinates in config.get(
            "waypoints_dict", {"goal": [5, 0]}
        ).items()
    }
    grid_w = int(config.get("grid_w", 12))
    grid_h = int(config.get("grid_h", 12))
    gamma = float(config.get("gamma", 0.99))
    goal_reward = float(config.get("goal_reward", 10000))

    automaton = LTLfAutomaton(formula)
    validation_report = validate_automaton(
        automaton, waypoints, width=grid_w, height=grid_h
    )
    print(
        "=== LTLf TRAINING (SB3 SAC) ===\n"
        f"Formula: {formula}\n"
        f"Waypoints: {waypoints}\n"
        f"DFA: states={automaton.states}, pre-trace={automaton.initial_state}, "
        f"accepting={sorted(automaton.accepting_states)}\n"
        "SAC action: Box(-1, 1, (1,)) -> floor-scaled LunarLander action.\n"
        "Gym reward is ignored by design.\n"
        f"{validation_report.format()}"
    )

    data_path = data_dir / "sac_data.npz"
    if not args.post_process:
        automaton.render_graph(directory=image_dir)
        abstract_mdp = LTLfWaypointMDP(
            waypoints_dict=waypoints,
            ltlf_automaton=automaton,
            width=grid_w,
            height=grid_h,
            gamma=gamma,
            goal_reward=goal_reward,
        )
        abstract_mdp.value_iteration()

        save_sequential_heatmaps(
            abstract_mdp,
            filename_prefix="sac_experiment",
            output_dir=image_dir / "heatmaps",
        )

        seeds = [args.seed + index for index in range(args.num_seeds)]
        seed_metrics = []
        for run_index, run_seed in enumerate(seeds, start=1):
            print(f"\n=== SEED RUN {run_index}/{args.num_seeds}: seed={run_seed} ===")
            base_env = gym.make("LunarLander-v3", continuous=False)
            continuous_env = DiscreteToContinuousActionWrapper(base_env)
            task_env = LTLfTaskWrapper(
                continuous_env,
                abstract_mdp,
                use_shaping=not args.no_shaping,
                shaping_scale=args.shaping_scale,
                goal_reward=goal_reward,
                training_shaping_gamma=args.training_shaping_gamma,
            )
            run_policy_dir = policy_dir if args.num_seeds == 1 else policy_dir / f"seed_{run_seed}"
            try:
                metrics = run_sequential_training(
                    env=task_env,
                    task_wrapper=task_env,
                    abstract_mdp=abstract_mdp,
                    episodes=args.episodes,
                    policy_dir=run_policy_dir,
                    log_file=log_dir / f"sac_training_seed_{run_seed}.log",
                    log_interval=args.log_interval,
                    learning_rate=args.learning_rate,
                    buffer_size=args.buffer_size,
                    learning_starts=args.learning_starts,
                    batch_size=args.batch_size,
                    tau=args.tau,
                    ent_coef=args.ent_coef,
                    seed=run_seed,
                    device=args.device,
                )
                seed_metrics.append(metrics)
                save_training_data(data_dir / f"sac_data_seed_{run_seed}.npz", **metrics)
            finally:
                task_env.close()
        save_training_data(data_path, **_aggregate_seed_metrics(seed_metrics, seeds))

    data = np.load(data_path, allow_pickle=False)
    plot_buffer_fractions(
        data["buffer_histories"],
        filename=plot_dir / "buffer_fractions_sac.png",
        window_size=args.plot_window,
        state_labels=data["automaton_states"],
    )
    plot_shaping_reward_breakdown(
        data["task_rewards"],
        data["learning_rewards"],
        data["entropy_coefficient_history"],
        window_size=args.plot_window,
        filename=plot_dir / "reward_breakdown_sac.png",
        exploration_label="Entropy coefficient (alpha)",
    )
    task_reward_runs = data["task_rewards_runs"] if "task_rewards_runs" in data else data["task_rewards"][np.newaxis, :]
    learning_reward_runs = data["learning_rewards_runs"] if "learning_rewards_runs" in data else data["learning_rewards"][np.newaxis, :]
    entropy_runs = data["entropy_coefficient_history_runs"] if "entropy_coefficient_history_runs" in data else data["entropy_coefficient_history"][np.newaxis, :]
    seed_values = data["seeds"] if "seeds" in data else np.asarray([args.seed])
    for run_seed, task_rewards, learning_rewards, entropy_history in zip(seed_values, task_reward_runs, learning_reward_runs, entropy_runs):
        plot_shaping_reward_breakdown(
            task_rewards,
            learning_rewards,
            entropy_history,
            window_size=args.plot_window,
            filename=plot_dir / f"reward_breakdown_sac_seed_{int(run_seed)}.png",
            exploration_label="Entropy coefficient (alpha)",
        )
    plot_training_variance(
        learning_reward_runs,
        window_size=args.plot_window,
        filename=plot_dir / "training_variance_sac.png",
    )
    print("\nFinished.")


# ==============================
# Command-line entry point
# ==============================

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="LTLf LunarLander training with Stable-Baselines3 SAC."
    )
    parser.add_argument("--experiment-name", type=_experiment_name, required=True, help="Output directory name under results/.")
    parser.add_argument("--episodes", type=int, default=1000)
    parser.add_argument("--num-seeds", type=_positive_int, default=1, help="Number of training runs with consecutive seeds.")
    parser.add_argument("--config", type=Path, default=SCRIPT_DIR / "trajectory.json")
    parser.add_argument("--shaping-scale", type=float, default=1.0)
    parser.add_argument("--log-interval", type=int, default=100)
    parser.add_argument("--plot-window", type=int, default=500)
    parser.add_argument("--learning-rate", type=float, default=3e-4)
    parser.add_argument("--buffer-size", type=int, default=300000)
    parser.add_argument("--learning-starts", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--tau", type=float, default=0.005)
    parser.add_argument(
        "--ent-coef",
        type=_parse_entropy_coefficient,
        default="auto",
        help="SAC entropy coefficient, for example 'auto', 'auto_0.1' or 0.05.",
    )
    parser.add_argument("--seed", type=int, default=42, help="First training seed.")
    parser.add_argument("--device", default="auto")
    parser.add_argument(
        "--training-shaping-gamma",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Use gamma*Phi(next)-Phi(state) during training; disable to use Phi(next)-Phi(state).",
    )
    parser.add_argument("--no-shaping", action="store_true")
    parser.add_argument("--post-process", action="store_true")
    main(parser.parse_args())


## 8. Write grid visualization and SAC evaluation modules


In [ ]:
%%writefile grid_overlay.py
"""Visualise the abstract grid on top of the LunarLander environment.

The conversion used here is the inverse of ``utils.phi_mapping_grid``.  Grid
cells in the resulting image therefore represent exactly the abstract states
used by the trainer, rather than an evenly spaced, screen-only decoration.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from utils import phi_mapping_grid, spatial_grid_boundaries


SCRIPT_DIR = Path(__file__).resolve().parent
DEFAULT_CONFIG = SCRIPT_DIR / "trajectory.json"


@dataclass(frozen=True)
class LunarLanderGeometry:
    """Screen/world constants required to project observations onto a frame."""

    viewport_width: int
    viewport_height: int
    scale: float
    helipad_y: float
    leg_down: float


def geometry_from_env(env: gym.Env) -> LunarLanderGeometry:
    """Read projection constants from a reset LunarLander environment."""
    from gymnasium.envs.box2d import lunar_lander

    base_env = env.unwrapped
    if not hasattr(base_env, "helipad_y"):
        raise TypeError("The supplied environment is not a LunarLander environment")
    return LunarLanderGeometry(
        viewport_width=lunar_lander.VIEWPORT_W,
        viewport_height=lunar_lander.VIEWPORT_H,
        scale=lunar_lander.SCALE,
        helipad_y=float(base_env.helipad_y),
        leg_down=lunar_lander.LEG_DOWN,
    )


def observation_to_pixel(observation: Sequence[float], geometry: LunarLanderGeometry) -> tuple[float, float]:
    """Project LunarLander's normalised (x, y) observation onto RGB pixels."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = (float(observation[0]) + 1.0) * half_world_width
    world_y = (
        float(observation[1]) * half_world_height
        + geometry.helipad_y
        + geometry.leg_down / geometry.scale
    )
    pixel_x = world_x * geometry.scale
    pixel_y = geometry.viewport_height - world_y * geometry.scale
    return pixel_x, pixel_y


def pixel_to_observation(pixel_x: float, pixel_y: float, geometry: LunarLanderGeometry) -> tuple[float, float]:
    """Invert the frame projection for the two discretised coordinates."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = float(pixel_x) / geometry.scale
    world_y = (geometry.viewport_height - float(pixel_y)) / geometry.scale
    observation_x = world_x / half_world_width - 1.0
    observation_y = (
        world_y - geometry.helipad_y - geometry.leg_down / geometry.scale
    ) / half_world_height
    return observation_x, observation_y


def _grid_boundaries(grid_w: int, grid_h: int, geometry: LunarLanderGeometry) -> tuple[np.ndarray, np.ndarray]:
    """Return the boundaries implied by the active spatial discretizer."""
    x_normalised, y_normalised = spatial_grid_boundaries(grid_w, grid_h)
    x_pixels = np.array(
        [observation_to_pixel((x, 0.0), geometry)[0] for x in x_normalised]
    )
    y_pixels = np.array(
        [observation_to_pixel((0.0, y), geometry)[1] for y in y_normalised]
    )

    # phi_mapping_grid clips observations outside its nominal domain into its
    # edge cells. Extend those cells to the RGB viewport edges whenever the
    # corresponding border observation maps to index 0 or to the last index.
    # Internal boundaries remain entirely inferred from the active mapper.
    left_observation_x, top_observation_y = pixel_to_observation(
        0.0, 0.0, geometry
    )
    right_observation_x, bottom_observation_y = pixel_to_observation(
        geometry.viewport_width, geometry.viewport_height, geometry
    )
    if phi_mapping_grid((left_observation_x, 0.0), grid_w, grid_h)[0] == 0:
        x_pixels[0] = min(x_pixels[0], 0.0)
    if phi_mapping_grid((right_observation_x, 0.0), grid_w, grid_h)[0] == grid_w - 1:
        x_pixels[-1] = max(x_pixels[-1], float(geometry.viewport_width))
    if phi_mapping_grid((0.0, bottom_observation_y), grid_w, grid_h)[1] == 0:
        y_pixels[0] = max(y_pixels[0], float(geometry.viewport_height))
    if phi_mapping_grid((0.0, top_observation_y), grid_w, grid_h)[1] == grid_h - 1:
        y_pixels[-1] = min(y_pixels[-1], 0.0)

    return x_pixels, y_pixels


def abstract_cell_to_pixel(grid_x: int, grid_y: int, grid_w: int, grid_h: int, geometry: LunarLanderGeometry) -> tuple[float, float]:
    """Return the pixel coordinates of an abstract cell's centre."""
    if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
        raise ValueError(f"Abstract cell ({grid_x}, {grid_y}) is outside the grid")
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    return (
        float((x_lines[grid_x] + x_lines[grid_x + 1]) / 2.0),
        float((y_lines[grid_y] + y_lines[grid_y + 1]) / 2.0),
    )


def draw_abstract_grid(frame: np.ndarray, geometry: LunarLanderGeometry, grid_w: int, grid_h: int, waypoints: Mapping[str, Sequence[int]] | None = None, observation: Sequence[float] | None = None, title: str = "LunarLander with Abstract Grid"):
    """Create a figure containing the effective clipped grid and its markers."""
    if grid_w < 2 or grid_h < 2:
        raise ValueError("grid_w and grid_h must both be at least 2")

    figure, axis = plt.subplots(figsize=(12, 8))
    axis.imshow(frame)
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    x_centres = (x_lines[:-1] + x_lines[1:]) / 2.0
    y_centres = (y_lines[:-1] + y_lines[1:]) / 2.0
    grid_color = "#ff1744"

    for x_pixel in x_lines:
        axis.axvline(x_pixel, color=grid_color, linewidth=1.6, alpha=0.95)
    for y_pixel in y_lines:
        axis.axhline(y_pixel, color=grid_color, linewidth=1.6, alpha=0.95)

    # The mapping clips everything outside its stated observation domain into
    # an edge cell; tint the currently occupied abstract cell when requested.
    if observation is not None:
        abstract_x, abstract_y = phi_mapping_grid(observation, grid_w, grid_h)
        x0, x1 = sorted((x_lines[abstract_x], x_lines[abstract_x + 1]))
        y0, y1 = sorted((y_lines[abstract_y], y_lines[abstract_y + 1]))
        x0, x1 = np.clip((x0, x1), 0, geometry.viewport_width)
        y0, y1 = np.clip((y0, y1), 0, geometry.viewport_height)
        axis.add_patch(
            Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                facecolor="#00e5ff",
                edgecolor="#00e5ff",
                linewidth=2.5,
                alpha=0.25,
                label=f"Current cell ({abstract_x}, {abstract_y})",
            )
        )

    for name, coordinates in (waypoints or {}).items():
        if len(coordinates) != 2:
            raise ValueError(f"Waypoint {name!r} must contain [x, y]")
        grid_x, grid_y = int(coordinates[0]), int(coordinates[1])
        if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
            raise ValueError(f"Waypoint {name!r} is outside the abstract grid")
        pixel_x, pixel_y = abstract_cell_to_pixel(
            grid_x, grid_y, grid_w, grid_h, geometry
        )
        axis.scatter(pixel_x, pixel_y, s=150, marker="o", color="#ffca28",
                     edgecolor="black", linewidth=1.3, zorder=5)
        axis.annotate(
            f"{name} ({grid_x}, {grid_y})",
            (pixel_x, pixel_y),
            xytext=(7, -10),
            textcoords="offset points",
            color="black",
            fontsize=9,
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.25", "fc": "#ffca28", "alpha": 0.9},
            zorder=6,
        )

    axis.set_xlim(0, geometry.viewport_width)
    # A small part of the configured y-domain can lie above the RGB viewport.
    # Keep it in view so that no abstract row or coordinate label disappears.
    visible_top = min(0.0, float(np.min(y_lines)))
    axis.set_ylim(geometry.viewport_height, visible_top)
    axis.set_title(title)

    # Put the abstract coordinates at cell centres. Since image coordinates
    # grow downwards while abstract y grows upwards, y_centres is descending:
    # label 0 consequently appears at the bottom and grid_h - 1 at the top.
    axis.set_xticks(x_centres, labels=range(grid_w))
    axis.set_yticks(y_centres, labels=range(grid_h))
    axis.set_xlabel("Abstract x-coordinate")
    axis.set_ylabel("Abstract y-coordinate")
    axis.tick_params(
        axis="both",
        which="major",
        color=grid_color,
        labelcolor=grid_color,
        labelsize=10,
        width=1.5,
        length=5,
    )
    for label in (*axis.get_xticklabels(), *axis.get_yticklabels()):
        label.set_fontweight("bold")

    if observation is not None:
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
    else:
        figure.tight_layout()
    return figure


def generate_overlay(output_path: str | Path, config_path: str | Path = DEFAULT_CONFIG, seed: int | None = 0) -> Path:
    """Reset LunarLander and save one annotated RGB frame as a PNG."""
    config_path = Path(config_path)
    with config_path.open(encoding="utf-8") as config_file:
        config = json.load(config_file)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    env = gym.make("LunarLander-v3", continuous=False, render_mode="rgb_array")
    try:
        observation, _ = env.reset(seed=seed)
        frame = env.render()
        geometry = geometry_from_env(env)
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=int(config.get("grid_w", 12)),
            grid_h=int(config.get("grid_h", 12)),
            waypoints=config.get("waypoints_dict", {}),
            observation=observation,
        )
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.close(figure)
    finally:
        env.close()
    return output_path.resolve()


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a LunarLander frame with the abstract grid overlaid."
    )
    parser.add_argument("--config", type=Path, default=DEFAULT_CONFIG)
    parser.add_argument("--output", type=Path)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()
    output_path = args.output or SCRIPT_DIR / "img" / "abstract_grid_overlay.png"
    saved_path = generate_overlay(output_path, args.config, args.seed)
    print(f"Image saved to: {saved_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Evaluate one or more LTLf-guided LunarLander SB3 SAC policies."""

# ==============================
# Standard library imports
# ==============================

import argparse
import json
import os
import sys
from pathlib import Path

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import SAC

from abstract_mdps import LTLfAutomaton, LTLfWaypointMDP
from agent import DiscreteToContinuousActionWrapper
from grid_overlay import (
    abstract_cell_to_pixel,
    draw_abstract_grid,
    geometry_from_env,
)
from utils import phi_mapping_sequential


# ==============================
# Paths and generic helpers
# ==============================

SCRIPT_DIR = Path(__file__).resolve().parent
EXPERIMENTS_DIR = SCRIPT_DIR.parent / "experiments"


def moving_average(data, window_size):
    """Return a moving average, or the original values when the window is larger."""
    values = np.asarray(data, dtype=np.float64)
    if len(values) < window_size:
        return values
    return np.convolve(values, np.ones(window_size) / window_size, mode="valid")


def _resolve_policy_path(policy, policy_dir):
    """Accept explicit paths as well as filenames relative to the policy directory."""
    supplied_path = Path(policy).expanduser()
    candidates = [supplied_path, Path(policy_dir).expanduser() / supplied_path]
    for candidate in list(candidates):
        if candidate.suffix != ".zip":
            candidates.append(candidate.with_suffix(".zip"))
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    raise FileNotFoundError(f"Policy '{policy}' not found either as an explicit path or under '{policy_dir}'.")


def _abstract_position(observation, q, grid_w, grid_h):
    """Map an environment observation to its abstract grid coordinates."""
    x, y, _ = phi_mapping_sequential(observation, q, grid_w, grid_h)
    return x, y


def _require_graphical_display():
    """Fail clearly when live rendering is requested from a headless Linux session."""
    if sys.platform.startswith("linux") and not (os.environ.get("DISPLAY") or os.environ.get("WAYLAND_DISPLAY")):
        raise RuntimeError("Live rendering requires DISPLAY or WAYLAND_DISPLAY. In Kaggle, SSH or another headless session, use --trace-grid instead.")


class MatplotlibFrameRenderer:
    """Display RGB frames without asking LunarLander to open an SDL window."""

    def __init__(self, frame, fps):
        _require_graphical_display()
        plt.ion()
        self.delay = 1.0 / max(float(fps), 1.0)
        self.figure, self.axis = plt.subplots(figsize=(9, 6))
        self.image = self.axis.imshow(frame)
        self.axis.set_title("LunarLander SAC policy")
        self.axis.axis("off")
        self.figure.tight_layout()
        plt.show(block=False)
        self.figure.canvas.draw_idle()
        self.figure.canvas.flush_events()

    def update(self, frame):
        """Replace the displayed frame and process pending GUI events."""
        if not plt.fignum_exists(self.figure.number):
            raise KeyboardInterrupt("The rendering window was closed")
        self.image.set_data(frame)
        self.figure.canvas.draw_idle()
        self.figure.canvas.flush_events()
        plt.pause(self.delay)

    def close(self):
        """Close only the figure owned by this renderer."""
        plt.close(self.figure)


# ==============================
# Policy evaluation
# ==============================

def evaluate_policy(policy, policy_dir, episodes, render, formula, waypoints_dict, goal_reward, grid_w, grid_h, seed, trace_episodes=0, device="auto", render_fps=60.0):
    """Load and evaluate one policy using the same DFA semantics as training."""
    # Rebuild the same automaton and abstract MDP used during training.
    policy_path = _resolve_policy_path(policy, policy_dir)
    policy_name = policy_path.name
    automaton = LTLfAutomaton(formula)
    abstract_mdp = LTLfWaypointMDP(waypoints_dict=waypoints_dict, ltlf_automaton=automaton, width=grid_w, height=grid_h, goal_reward=goal_reward)
    automaton_states = list(automaton.states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}

    # Keep LunarLander discrete internally while exposing the Box action space
    # used when the SAC policy was trained.
    if render:
        _require_graphical_display()
    render_mode = "rgb_array" if render or trace_episodes else None
    base_env = gym.make(
        "LunarLander-v3", continuous=False, render_mode=render_mode
    )
    env = DiscreteToContinuousActionWrapper(base_env)

    # SB3 checkpoints include the complete actor, critics and hyperparameters.
    try:
        model = SAC.load(policy_path, device=device)
    except Exception:
        env.close()
        raise

    task_returns = []
    environment_returns = []
    episode_lengths = []
    successes = 0
    state_reach_counts = {q: 0 for q in automaton_states}
    grid_traces = []
    trace_frames = []
    trace_geometries = []
    frame_renderer = None

    # Run every requested episode sequentially.
    try:
        for episode in range(episodes):
            episode_seed = None if seed is None else seed + episode
            observation, _ = env.reset(seed=episode_seed)
            tracing = episode < trace_episodes
            initial_frame = env.render() if render or tracing else None
            if render:
                if frame_renderer is None:
                    frame_renderer = MatplotlibFrameRenderer(initial_frame, render_fps)
                else:
                    frame_renderer.update(initial_frame)
            if tracing:
                trace_frames.append(initial_frame)
                trace_geometries.append(geometry_from_env(env))
                initial_cell = _abstract_position(observation, automaton.get_initial_q(), grid_w, grid_h)
                cell_trace = [initial_cell]

            # Training consumes the valuation at s0 before choosing the first action.
            initial_q = automaton.get_initial_q()
            initial_x, initial_y = _abstract_position(observation, initial_q, grid_w, grid_h)
            initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
            q = automaton.get_next_q(initial_q, initial_truth_assignment)
            if q not in state_to_index:
                raise RuntimeError(f"DFA returned unknown state {q!r}")

            reached_states = {q}
            success = automaton.is_goal_reached(q)
            terminated = truncated = False
            environment_return = 0.0
            steps = 0

            while not (success or terminated or truncated):
                # Append the current DFA state as a one-hot vector.
                one_hot = np.zeros(len(automaton_states), dtype=np.float32)
                one_hot[state_to_index[q]] = 1.0
                augmented_state = np.concatenate((observation, one_hot)).astype(np.float32)

                # Deterministic prediction uses the mean SAC actor action.
                action, _ = model.predict(augmented_state, deterministic=True)

                next_observation, env_reward, terminated, truncated, _ = env.step(action)
                environment_return += float(env_reward)
                steps += 1

                # Advance the DFA using the propositions true in the arrival state.
                x, y = _abstract_position(next_observation, q, grid_w, grid_h)
                if tracing and (x, y) != cell_trace[-1]:
                    cell_trace.append((x, y))
                truth_assignment = abstract_mdp._get_truth_assignment(x, y)
                next_q = automaton.get_next_q(q, truth_assignment)
                if next_q not in state_to_index:
                    raise RuntimeError(f"DFA returned unknown state {next_q!r}")

                # Report every effective DFA transition during evaluation.
                if next_q != q:
                    if automaton.is_goal_reached(next_q):
                        print(f"[{policy_name} | Episode {episode + 1}] DFA transition {q} -> {next_q}: final goal reached.")
                    else:
                        print(f"[{policy_name} | Episode {episode + 1}] DFA transition {q} -> {next_q}: intermediate waypoint reached.")

                reached_states.add(next_q)

                observation = next_observation
                q = next_q
                success = automaton.is_goal_reached(q)

                if render:
                    frame_renderer.update(env.render())

            # Store episode-level metrics and count every DFA state reached at least once.
            successes += int(success)
            for reached_q in reached_states:
                state_reach_counts[reached_q] += 1
            task_returns.append(float(goal_reward) if success else 0.0)
            environment_returns.append(environment_return)
            episode_lengths.append(steps)
            if tracing:
                grid_traces.append(cell_trace)
    finally:
        if frame_renderer is not None:
            frame_renderer.close()
        env.close()

    return {
        "policy": policy_name,
        "path": str(policy_path),
        "task_returns": task_returns,
        "environment_returns": environment_returns,
        "episode_lengths": episode_lengths,
        "successes": successes,
        "state_reach_counts": state_reach_counts,
        "grid_traces": grid_traces,
        "trace_frames": trace_frames,
        "trace_geometries": trace_geometries,
    }


# ==============================
# Plotting helpers
# ==============================

def _safe_stem(name):
    """Create a filesystem-safe plot stem from a checkpoint filename."""
    return "".join(character if character.isalnum() or character in "-_." else "_" for character in Path(name).stem)


def plot_policy(result, window_size, output_dir):
    """Plot Gym returns for one policy."""
    returns = result["environment_returns"]
    smooth = moving_average(returns, window_size)

    plt.figure(figsize=(10, 6))
    plt.plot(returns, alpha=0.3, color="gray", label="Raw Gym return")
    start = window_size - 1 if len(returns) >= window_size else 0
    plt.plot(range(start, start + len(smooth)), smooth, color="blue", linewidth=2, label=f"Moving average (window={window_size})")
    plt.title(f"Evaluation: {result['policy']}")
    plt.xlabel("Episode")
    plt.ylabel("Gym return")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    output_path = output_dir / f"eval_{_safe_stem(result['policy'])}.png"
    plt.savefig(output_path, bbox_inches="tight")
    plt.close()
    return output_path


def plot_comparison(results, window_size, output_dir):
    """Plot smoothed Gym returns for multiple policies."""
    plt.figure(figsize=(12, 6))
    for result in results:
        returns = result["environment_returns"]
        smooth = moving_average(returns, window_size)
        start = window_size - 1 if len(returns) >= window_size else 0
        plt.plot(range(start, start + len(smooth)), smooth, linewidth=2, label=result["policy"])
    plt.title("Policy comparison")
    plt.xlabel("Episode")
    plt.ylabel("Gym return")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    output_path = output_dir / "policy_comparison.png"
    plt.savefig(output_path, bbox_inches="tight")
    plt.close()
    return output_path


def plot_grid_traces(result, waypoints_dict, grid_w, grid_h, output_dir):
    """Save one abstract-grid path image for every recorded episode."""
    output_paths = []
    trace_data = zip(
        result["grid_traces"],
        result["trace_frames"],
        result["trace_geometries"],
    )
    for episode_index, (cells, frame, geometry) in enumerate(trace_data, start=1):
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=grid_w,
            grid_h=grid_h,
            waypoints=waypoints_dict,
            title=f"Agent Abstract-Cell Trace — Episode {episode_index}",
        )
        axis = figure.axes[0]
        points = [
            abstract_cell_to_pixel(x, y, grid_w, grid_h, geometry)
            for x, y in cells
        ]
        if points:
            pixel_x, pixel_y = zip(*points)
            axis.plot(
                pixel_x,
                pixel_y,
                color="#00e5ff",
                linewidth=2.8,
                marker="o",
                markersize=5,
                label="Visited-cell path",
                zorder=4,
            )
            for change_index, ((cell_x, cell_y), (point_x, point_y)) in enumerate(
                zip(cells, points)
            ):
                axis.annotate(
                    str(change_index),
                    (point_x, point_y),
                    ha="center",
                    va="center",
                    fontsize=7,
                    fontweight="bold",
                    color="black",
                    zorder=7,
                )
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
        output_path = output_dir / (
            f"grid_trace_{_safe_stem(result['policy'])}_episode_{episode_index}.png"
        )
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.close(figure)
        output_paths.append(output_path)
    return output_paths


def format_waypoint_trace(cells, waypoints_dict):
    """Report the first cell-change index at which each waypoint was visited."""
    first_visit = {}
    for index, cell in enumerate(cells):
        first_visit.setdefault(tuple(cell), index)
    return ", ".join(
        f"{name}=reached@{first_visit[tuple(position)]}"
        if tuple(position) in first_visit
        else f"{name}=missed"
        for name, position in waypoints_dict.items()
    )


# ==============================
# Command-line interface
# ==============================

def _positive_int(value):
    """Parse and validate a strictly positive integer."""
    parsed = int(value)
    if parsed <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return parsed


def _positive_float(value):
    """Parse and validate a strictly positive floating-point value."""
    parsed = float(value)
    if parsed <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return parsed


def _select_files_graphically(policy_dir, config_path):
    """Select policy checkpoints and the experiment configuration with native dialogs."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError as error:
        raise RuntimeError(
            "The graphical selector requires tkinter. Install python3-tk or pass "
            "the policy paths and --config from the command line."
        ) from error

    try:
        root = tk.Tk()
    except tk.TclError as error:
        raise RuntimeError(
            "The graphical selector could not be opened. Make sure a desktop "
            "session is available, or use the command-line arguments."
        ) from error
    root.withdraw()
    root.update()

    try:
        initial_directory = EXPERIMENTS_DIR if EXPERIMENTS_DIR.is_dir() else SCRIPT_DIR
        policies = filedialog.askopenfilenames(
            parent=root,
            title="Select one or more policy files",
            initialdir=str(initial_directory),
            filetypes=[
                ("Stable-Baselines3 checkpoints", "*.zip"),
                ("All files", "*"),
            ],
        )
        if not policies:
            raise RuntimeError("No policy file was selected.")

        config = filedialog.askopenfilename(
            parent=root,
            title="Select trajectory.json",
            initialdir=str(initial_directory),
            initialfile=Path(config_path).name,
            filetypes=[
                ("JSON files", "*.json"),
                ("All files", "*"),
            ],
        )
        if not config:
            raise RuntimeError("No trajectory configuration was selected.")
    finally:
        root.destroy()

    return list(policies), Path(config)


def parse_args():
    """Build and parse the evaluator command-line arguments."""
    parser = argparse.ArgumentParser(description="Evaluate LTLf-guided SAC policies for LunarLander.")
    parser.add_argument(
        "policies",
        nargs="*",
        help="Checkpoint filenames or explicit checkpoint paths. If omitted, graphical file selectors are opened.",
    )
    parser.add_argument("--config", type=Path, default=SCRIPT_DIR / "trajectory.json", help="Experiment JSON configuration.")
    parser.add_argument("--policy-dir", type=Path, default=SCRIPT_DIR / "policy", help="Directory used to resolve checkpoint filenames.")
    parser.add_argument("--gui", action="store_true", help="Select policies and trajectory.json using graphical dialogs.")
    parser.add_argument("--episodes", type=_positive_int, default=100)
    parser.add_argument("--window", type=_positive_int, default=10)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--render", action="store_true", help="Display RGB frames live through Matplotlib, avoiding LunarLander's native SDL window.")
    parser.add_argument("--render-fps", type=_positive_float, default=60.0, help="Target live-rendering frame rate (default: 60).")
    parser.add_argument(
        "--trace-grid",
        action="store_true",
        help="Save the sequence of abstract cells visited during evaluation.",
    )
    parser.add_argument(
        "--trace-episodes",
        type=_positive_int,
        default=1,
        help="Number of episodes to trace when --trace-grid is enabled (default: 1).",
    )
    parser.add_argument("--output-dir", type=Path, default=SCRIPT_DIR / "img" / "evaluation")
    return parser.parse_args()


# ==============================
# Main program
# ==============================

def main():
    """Load the configuration, evaluate the policies, and generate the plots."""
    args = parse_args()
    if args.render and args.trace_grid:
        raise SystemExit(
            "--render and --trace-grid cannot be used together because Gymnasium "
            "requires a single render mode. Run them as separate evaluations."
        )

    # Open native file dialogs when requested or when no policy was supplied.
    if args.gui or not args.policies:
        try:
            args.policies, args.config = _select_files_graphically(args.policy_dir, args.config)
        except RuntimeError as error:
            raise SystemExit(f"Selection cancelled: {error}") from error

    # Load the LTLf task shared with the trainer.
    with args.config.expanduser().open(encoding="utf-8") as config_file:
        config = json.load(config_file)

    formula = config["formula"]
    raw_waypoints = config["waypoints_dict"]
    waypoints_dict = {name: tuple(coordinates) for name, coordinates in raw_waypoints.items()}
    grid_w = int(config.get("grid_w", 12))
    grid_h = int(config.get("grid_h", 12))
    goal_reward = float(config.get("goal_reward", 10000.0))

    # Evaluate policies one at a time to keep rendering and output deterministic.
    results = []
    for policy in args.policies:
        traced_episodes = min(args.trace_episodes, args.episodes) if args.trace_grid else 0
        result = evaluate_policy(
            policy, args.policy_dir, args.episodes, args.render, formula,
            waypoints_dict, goal_reward, grid_w, grid_h, args.seed,
            trace_episodes=traced_episodes,
            device=args.device,
            render_fps=args.render_fps,
        )
        results.append(result)

    # Print the summary and create one plot for each evaluated policy.
    args.output_dir.mkdir(parents=True, exist_ok=True)
    for result in results:
        success_rate = result["successes"] / args.episodes
        mean_gym_return = np.mean(result["environment_returns"])
        mean_length = np.mean(result["episode_lengths"])
        reached = ", ".join(f"q={q}: {count}/{args.episodes}" for q, count in result["state_reach_counts"].items())
        print(f"[{result['policy']}] success={success_rate:.1%}, mean Gym return={mean_gym_return:.2f}, mean length={mean_length:.1f} | reached: {reached}")
        print(f"Plot saved to: {plot_policy(result, args.window, args.output_dir)}")
        if args.trace_grid:
            trace_paths = plot_grid_traces(
                result, waypoints_dict, grid_w, grid_h, args.output_dir
            )
            for episode_index, (cells, trace_path) in enumerate(
                zip(result["grid_traces"], trace_paths), start=1
            ):
                waypoint_status = format_waypoint_trace(cells, waypoints_dict)
                print(
                    f"Grid trace episode {episode_index}: {waypoint_status} | "
                    f"saved to: {trace_path}"
                )

    # Add a combined comparison when more than one policy was requested.
    if len(results) > 1:
        print(f"Comparison saved to: {plot_comparison(results, args.window, args.output_dir)}")


if __name__ == "__main__":
    main()


## 9. Write the LTLf task configuration


In [ ]:
%%writefile trajectory.json
{
    "formula": "F(wp1 & X(F(g1)))",
    "grid_w": 12,
    "grid_h": 12,
    "goal_reward": 10000,
    "waypoints_dict": {
        "wp1": [1, 8],
        "g1": [8, 8]
    }
}

## 10. Configure the training run

The Gym reward is ignored by the trainer. `ENTROPY_COEFFICIENT` accepts `"auto"`, `"auto_0.1"` or a positive fixed number.


In [ ]:
EPISODES = 1000
NUM_SEEDS = 1
SHAPING_SCALE = 1.0
TRAINING_USE_GAMMA = True
LOG_INTERVAL = 100
PLOT_WINDOW = 500
LEARNING_RATE = 3e-4
BUFFER_SIZE = 300_000
LEARNING_STARTS = 100
BATCH_SIZE = 256
TAU = 0.005
ENTROPY_COEFFICIENT = "auto"
DISABLE_SHAPING = False
SEED = 42
DEVICE = "auto"

print(f"Episodes: {EPISODES}")
print(f"Number of seeds: {NUM_SEEDS}")
print(f"Shaping enabled: {not DISABLE_SHAPING}")
print(f"Shaping scale: {SHAPING_SCALE}")
print(f"Training uses gamma: {TRAINING_USE_GAMMA}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Replay buffer size: {BUFFER_SIZE}")
print(f"Entropy coefficient: {ENTROPY_COEFFICIENT}")
print(f"Seed: {SEED}")
print(f"Device: {DEVICE}")


## 11. Run training


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "trainer.py",
    "--episodes", str(EPISODES),
    "--config", "trajectory.json",
    "--shaping-scale", str(SHAPING_SCALE),
    "--log-interval", str(LOG_INTERVAL),
    "--plot-window", str(PLOT_WINDOW),
    "--learning-rate", str(LEARNING_RATE),
    "--buffer-size", str(BUFFER_SIZE),
    "--learning-starts", str(LEARNING_STARTS),
    "--batch-size", str(BATCH_SIZE),
    "--tau", str(TAU),
    "--ent-coef", str(ENTROPY_COEFFICIENT),
    "--device", DEVICE,
]
command.extend(["--num-seeds", str(NUM_SEEDS), "--seed", str(SEED)])
if not TRAINING_USE_GAMMA:
    command.append("--no-training-shaping-gamma")
if DISABLE_SHAPING:
    command.append("--no-shaping")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
environment["PYTHONUNBUFFERED"] = "1"
if SEED is not None:
    environment["PYTHONHASHSEED"] = str(SEED)

print("Running:", " ".join(command))
process = subprocess.Popen(command, cwd=WORK_DIR, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)


## 12. Inspect saved metrics, logs, heatmaps, and policies


In [ ]:
import numpy as np

data_path = WORK_DIR / "results" / "sac_data.npz"
metrics = np.load(data_path, allow_pickle=False)

print("Saved metrics:")
for key in metrics.files:
    value = metrics[key]
    print(f"- {key}: shape={value.shape}, dtype={value.dtype}")

print(f"\nBest policy episode: {int(metrics['best_policy_episode'])}")
print(f"Best mean learning reward: {float(metrics['best_mean_learning_reward']):.3f}")
print(f"Overall success rate: {metrics['successes'].mean():.2%}")

log_path = WORK_DIR / "logs" / "sac_training.log"
if log_path.exists():
    print("\nLast log lines:\n")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-30:]))

print("\nGenerated heatmaps:")
for heatmap_path in sorted((WORK_DIR / "img" / "heatmaps").glob("*.png")):
    print(f"- {heatmap_path.relative_to(WORK_DIR)}")

print("\nSaved policies:")
for policy_path in sorted((WORK_DIR / "policy").glob("*.zip")):
    print(f"- {policy_path.name}")


## 13. Display generated plots


In [ ]:
from IPython.display import display
from PIL import Image

plot_paths = sorted((WORK_DIR / "img").glob("*.png"))
heatmap_paths = sorted((WORK_DIR / "img" / "heatmaps").glob("*.png"))
for image_path in plot_paths + heatmap_paths:
    print(image_path.relative_to(WORK_DIR))
    display(Image.open(image_path))


## 14. Package outputs for download


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/lunar_lander_dsac_outputs.zip")
output_directories = ("results", "img", "logs", "policy")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in output_directories:
        output_directory = WORK_DIR / directory_name
        if output_directory.exists():
            for output_path in sorted(output_directory.rglob("*")):
                if output_path.is_file():
                    archive.write(output_path, output_path.relative_to(WORK_DIR))
    configuration_path = WORK_DIR / "trajectory.json"
    if configuration_path.is_file():
        archive.write(configuration_path, configuration_path.name)

print(f"Output archive ready: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")
